# 02 Preprocessing and EDA RESET
Loads the full `MetroPT3(AirCompressor).csv`, validates the date range, saves a cleaned dataset, tables and EDA figures.



In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print('Python:', sys.executable)
print('pandas:', pd.__version__)



Python: c:\Users\user\Desktop\metropt_predictive_maintenance_starter\.venv\Scripts\python.exe
pandas: 2.2.2


In [2]:
current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent if current_dir.name == 'notebooks' else current_dir
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_TABLES_DIR = PROJECT_ROOT / 'outputs' / 'tables'
OUTPUT_FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures'
for p in [PROCESSED_DATA_DIR, OUTPUT_TABLES_DIR, OUTPUT_FIGURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)
raw_path = RAW_DATA_DIR / 'MetroPT3(AirCompressor).csv'
cleaned_path = PROCESSED_DATA_DIR / 'metropt3_cleaned.csv'
print('Raw path:', raw_path)



Raw path: c:\Users\user\Desktop\metropt_predictive_maintenance_starter\metropt_predictive_maintenance_starter\notebooks\metropt_reset_steps_02_to_06\data\raw\MetroPT3(AirCompressor).csv


In [6]:
if not raw_path.exists():
    print('Available raw CSV files:', [f.name for f in RAW_DATA_DIR.glob('*.csv')])
    raise FileNotFoundError(f'MetroPT3(AirCompressor).csv not found in {RAW_DATA_DIR}. Available files: {[f.name for f in RAW_DATA_DIR.glob("*.csv")]}')
df = pd.read_csv(raw_path)
print('Raw shape:', df.shape)
print('Columns:', df.columns.tolist())



Available raw CSV files: []


FileNotFoundError: MetroPT3(AirCompressor).csv not found in c:\Users\user\Desktop\metropt_predictive_maintenance_starter\metropt_predictive_maintenance_starter\notebooks\metropt_reset_steps_02_to_06\data\raw. Available files: []

In [ ]:
timestamp_col = 'timestamp' if 'timestamp' in df.columns else None
if timestamp_col is None:
    for c in ['Timestamp','time','Time','datetime','Datetime','date','Date']:
        if c in df.columns:
            timestamp_col = c; break
if timestamp_col is None:
    raise ValueError('Timestamp column not found')
df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors='coerce')
invalid_ts = int(df[timestamp_col].isna().sum())
df = df.dropna(subset=[timestamp_col]).sort_values(timestamp_col).reset_index(drop=True)
duplicate_timestamps = int(df.duplicated(subset=[timestamp_col]).sum())
print('Timestamp column:', timestamp_col)
print('Invalid timestamps removed:', invalid_ts)
print('Duplicate timestamps:', duplicate_timestamps)
print('Dataset start:', df[timestamp_col].min())
print('Dataset end:', df[timestamp_col].max())
print('Rows by month:')
print(df[timestamp_col].dt.to_period('M').value_counts().sort_index())
if df[timestamp_col].max() < pd.Timestamp('2020-07-01'):
    raise ValueError('Partial dataset detected. The raw data ends before July 2020.')



In [ ]:
numerical_sensors = ['TP2','TP3','H1','DV_pressure','Reservoirs','Oil_temperature','Motor_current','Caudal_impulses']
digital_sensors = ['COMP','DV_eletric','Towers','MPG','LPS','Pressure_switch','Oil_level']
available_num = [c for c in numerical_sensors if c in df.columns]
available_dig = [c for c in digital_sensors if c in df.columns]
for c in available_num + available_dig:
    df[c] = pd.to_numeric(df[c], errors='coerce')
print('Numerical sensors:', available_num)
print('Digital sensors:', available_dig)



In [ ]:
missing_before = df.isna().sum().reset_index()
missing_before.columns = ['column','missing_before']
missing_total_before = int(missing_before['missing_before'].sum())
df_clean = df.ffill().dropna().reset_index(drop=True)
missing_after = df_clean.isna().sum().reset_index()
missing_after.columns = ['column','missing_after']
missing_summary = missing_before.merge(missing_after, on='column', how='left').fillna(0)
missing_summary['missing_before_percentage'] = missing_summary['missing_before'] / len(df) * 100
missing_summary['missing_after_percentage'] = missing_summary['missing_after'] / len(df_clean) * 100
missing_summary.to_csv(OUTPUT_TABLES_DIR / 'missing_values_before_after.csv', index=False)
print('Cleaned shape:', df_clean.shape)
print('Cleaned end:', df_clean[timestamp_col].max())



In [ ]:
df_clean.to_csv(cleaned_path, index=False)
print('Saved cleaned dataset:', cleaned_path)



In [ ]:
failure_events = pd.DataFrame({
    'event_id':['F1','F2','F3','F4'],
    'failure_type':['Air leak','Air leak','Air leak','Air leak'],
    'failure_start':['2020-04-18 00:00','2020-05-29 23:30','2020-06-05 10:00','2020-07-15 14:30'],
    'failure_end':['2020-04-18 23:59','2020-05-30 06:00','2020-06-07 14:30','2020-07-15 19:00']
})
failure_events['failure_start'] = pd.to_datetime(failure_events['failure_start'])
failure_events['failure_end'] = pd.to_datetime(failure_events['failure_end'])
failure_events['inside_dataset_range'] = (failure_events['failure_start'].ge(df_clean[timestamp_col].min()) & failure_events['failure_end'].le(df_clean[timestamp_col].max()))
failure_events['records_in_failure_interval'] = [int(((df_clean[timestamp_col] >= r.failure_start) & (df_clean[timestamp_col] <= r.failure_end)).sum()) for _, r in failure_events.iterrows()]
failure_events.to_csv(OUTPUT_TABLES_DIR / 'documented_failure_events_checked.csv', index=False)
failure_events



In [ ]:
preprocessing_summary = pd.DataFrame({'item':['Raw rows','Raw columns','Cleaned rows','Cleaned columns','Invalid timestamps removed','Duplicate timestamps','Missing before','Missing after','Dataset start','Dataset end'], 'value':[df.shape[0],df.shape[1],df_clean.shape[0],df_clean.shape[1],invalid_ts,duplicate_timestamps,missing_total_before,int(df_clean.isna().sum().sum()),df_clean[timestamp_col].min(),df_clean[timestamp_col].max()]})
preprocessing_summary.to_csv(OUTPUT_TABLES_DIR / 'preprocessing_summary.csv', index=False)
df_clean[available_num].describe().T.to_csv(OUTPUT_TABLES_DIR / 'sensor_summary_statistics.csv')
df_clean[available_num].corr().to_csv(OUTPUT_TABLES_DIR / 'correlation_matrix.csv')
preprocessing_summary



In [ ]:
plot_hourly = df_clean[[timestamp_col]+available_num].set_index(timestamp_col).resample('1h').mean()
for sensor in available_num:
    plt.figure(figsize=(12,4))
    plt.plot(plot_hourly.index, plot_hourly[sensor])
    plt.title(f'{sensor} hourly average')
    plt.xlabel('Time'); plt.ylabel(sensor)
    plt.tight_layout(); plt.savefig(OUTPUT_FIGURES_DIR / f'{sensor}_hourly_average.png', dpi=300); plt.show()



In [ ]:
corr = df_clean[available_num].corr()
plt.figure(figsize=(9,7))
plt.imshow(corr, aspect='auto')
plt.colorbar(label='Correlation')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
plt.yticks(range(len(corr.index)), corr.index)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        plt.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.title('Correlation heatmap')
plt.tight_layout(); plt.savefig(OUTPUT_FIGURES_DIR / 'correlation_heatmap.png', dpi=300); plt.show()



In [ ]:
sample = df_clean.sample(n=min(200000, len(df_clean)), random_state=42)
for sensor in available_num:
    plt.figure(figsize=(8,4))
    plt.hist(sample[sensor].dropna(), bins=60)
    plt.title(f'Distribution of {sensor}')
    plt.xlabel(sensor); plt.ylabel('Frequency')
    plt.tight_layout(); plt.savefig(OUTPUT_FIGURES_DIR / f'{sensor}_distribution.png', dpi=300); plt.show()



In [ ]:
key = [c for c in ['TP2','TP3','Oil_temperature','Motor_current'] if c in df_clean.columns]
for _, event in failure_events.iterrows():
    window = df_clean[(df_clean[timestamp_col] >= event.failure_start - pd.Timedelta(hours=24)) & (df_clean[timestamp_col] <= event.failure_end)][[timestamp_col]+key]
    if window.empty: continue
    window = window.set_index(timestamp_col).resample('1min').mean().reset_index()
    for sensor in key:
        plt.figure(figsize=(12,4))
        plt.plot(window[timestamp_col], window[sensor])
        plt.axvline(event.failure_start, linestyle='--', label='Failure start')
        plt.axvline(event.failure_end, linestyle='--', label='Failure end')
        plt.title(f'{sensor} around {event.event_id}')
        plt.xlabel('Time'); plt.ylabel(sensor); plt.legend()
        plt.tight_layout(); plt.savefig(OUTPUT_FIGURES_DIR / f'{event.event_id}_{sensor}_failure_window.png', dpi=300); plt.show()



In [ ]:
checklist = pd.DataFrame({'task':['Full raw data loaded','Cleaned dataset saved','Dataset reaches July or later','F1-F4 events checked','EDA figures saved'], 'status':['Complete','Complete','Complete' if df_clean[timestamp_col].max() >= pd.Timestamp('2020-07-01') else 'Check required','Complete','Complete']})
checklist.to_csv(OUTPUT_TABLES_DIR / 'preprocessing_eda_completion_checklist.csv', index=False)
print('Step 2 complete. Cleaned shape:', df_clean.shape)
